# Lección 2: ARIMA — Nuestro primer modelo de predicción

**Dataset**: GEFCom2014 — 3 años de carga eléctrica horaria

**Objetivo**: Construir un modelo SARIMAX y predecir la carga eléctrica 3 horas adelante.

---

## ¿Qué es ARIMA?

ARIMA = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage

Desglose:
- **AR (AutoRegresivo)**: Usa valores PASADOS para predecir el futuro
- **I (Integrated)**: Diferencia los datos para hacerlos estacionarios
- **MA (Moving Average)**: Usa ERRORES PASADOS para corregir predicciones

Como我们的 serie tiene estacionalidad, usamos **SARIMA** (Seasonal ARIMA) con parámetros adicionales: `P, D, Q, m`.

## 1. Imports

In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt
import math

from pandas.plotting import autocorrelation_plot
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from common.utils import load_data, mape
from IPython.display import Image

%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format
np.set_printoptions(precision=2)
warnings.filterwarnings('ignore')

print('✅ Imports listos — SARIMAX, MinMaxScaler, mape')

## 2. Cargar los datos

In [ ]:
energy = load_data('./data')[['load']]
print(f'Filas: {energy.shape[0]}')
print(f'Columnas: {list(energy.columns)}')
print(f'Rango: {energy.index.min()} → {energy.index.max()}')
energy.head(10)

In [ ]:
energy.plot(y='load', subplots=True, figsize=(15, 8), fontsize=12)
plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.suptitle('Serie completa: Ene 2012 → Dic 2014', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Separar Train y Test (split TEMPORAL)

**Regla de oro en series temporales**: NUNCA hagas random split.

¿Por qué? Porque el tiempo va en una dirección. Si mezclas datos de 2012 con 2014 en el train, estás **haciendo trampa** (el modelo "ve" el futuro).

```
Train: Nov 1 → Dic 29, 2014  (1416 horas)
Test:  Dic 30 → Dic 31, 2014 (48 horas = 2 días)
```

In [ ]:
train_start_dt = '2014-11-01 00:00:00'
test_start_dt = '2014-12-30 00:00:00'

print(f'Train: {train_start_dt} → {test_start_dt}')
print(f'Test:  {test_start_dt} → 2014-12-31 23:00:00')

In [ ]:
# Visualizar train vs test
energy[(energy.index < test_start_dt) & (energy.index >= train_start_dt)][['load']]\
    .rename(columns={'load':'train'})\
    .join(energy[test_start_dt:][['load']].rename(columns={'load':'test'}), how='outer')\
    .plot(y=['train', 'test'], figsize=(15, 8), fontsize=12)

plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title('Train (naranja) vs Test (azul)', fontsize=14)
plt.legend(fontsize=12)
plt.show()

## 4. Escalar los datos (MinMaxScaler)

¿Por qué escalar? ARIMA funciona mejor cuando los datos están en el rango [0, 1].

- `fit_transform()` en train: aprende el rango y escala
- `transform()` en test: usa el mismo rango del train (no "ve" el futuro)

In [ ]:
train = energy.copy()[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][['load']]
test = energy.copy()[energy.index >= test_start_dt][['load']]

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

In [ ]:
# Escalar train
scaler = MinMaxScaler()
train['load'] = scaler.fit_transform(train)

# Verificar rango [0, 1]
print(f'Train min: {train["load"].min():.4f}')
print(f'Train max: {train["load"].max():.4f}')
train.head(10)

In [ ]:
# Comparar: original vs escalado
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

energy[(energy.index >= train_start_dt) & (energy.index < test_start_dt)][['load']]\
    .rename(columns={'load':'original'})\
    .plot.hist(bins=100, ax=axes[0], title='Original', fontsize=12)

train.rename(columns={'load':'escalado [0,1]'})\
    .plot.hist(bins=100, ax=axes[1], title='Escalado', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Escalar test (con el mismo scaler del train)
test['load'] = scaler.transform(test)
print(f'Test min: {test["load"].min():.4f}')
print(f'Test max: {test["load"].max():.4f}')
test.head()

## 5. Configurar SARIMAX

### Los 6 parámetros clave

| Parámetro | Qué controla | Ejemplo |
|-----------|-------------|---------|
| `p` | Lags auto-regresivos (cuántos valores pasados mira) | 4 = mira 4 horas atrás |
| `d` | Diferenciación (cuántas veces restar el anterior) | 1 = una vez |
| `q` | Lags de media móvil (cuántos errores pasados usa) | 0 = ninguno |
| `P` | Lags estacionales auto-regresivos | 1 = mira 24h atrás |
| `D` | Diferenciación estacional | 1 = una vez |
| `Q` | Lags estacionales de media móvil | 0 = ninguno |
| `m` | Período estacional | 24 = cada 24 horas |

En nuestro caso: `order=(4,1,0)` y `seasonal_order=(1,1,0,24)`

In [ ]:
HORIZON = 3  # Predecir 3 horas adelante
print(f'Forecasting horizon: {HORIZON} horas')

order = (4, 1, 0)           # p=4, d=1, q=0
seasonal_order = (1, 1, 0, 24)  # P=1, D=1, Q=0, m=24

print(f'order: {order}')
print(f'seasonal_order: {seasonal_order}')

### Entender los parámetros

**`order=(4, 1, 0)`**:
- `p=4`: El modelo mira las últimas 4 horas para predecir
- `d=1`: Diferencia una vez para quitar tendencia
- `q=0`: No usa errores pasados (solo AR)

**`seasonal_order=(1, 1, 0, 24)`**:
- `P=1`: Mira el valor de hace 24 horas (1 día)
- `D=1`: Diferencia estacional una vez
- `Q=0`: No usa errores estacionales
- `m=24`: El período es 24 horas (estacionalidad diaria)

In [ ]:
# Entrenar el modelo
model = SARIMAX(endog=train, order=order, seasonal_order=seasonal_order)
results = model.fit()

print(results.summary())

## 6. Walk-Forward Validation

¿Qué es? **Re-entrenar el modelo con cada nuevo dato**.

```
Paso 1: Entrena en [horas 1-720] → Predice hora 721
Paso 2: Entrena en [horas 2-721] → Predice hora 722
Paso 3: Entrena en [horas 3-722] → Predice hora 723
...
```

¿Por qué? Porque en producción, cada hora llega un dato nuevo y queremos la mejor predicción posible.

In [ ]:
# Preparar test data con columnas shifted para walk-forward
test_shifted = test.copy()

for t in range(1, HORIZON+1):
    test_shifted['load+'+str(t)] = test_shifted['load'].shift(-t, freq='H')

test_shifted = test_shifted.dropna(how='any')
print(f'Test shifted shape: {test_shifted.shape}')
test_shifted.head(5)

### ¿Qué es `test_shifted`?

Cada fila tiene:
- `load`: valor actual (target)
- `load+1`: valor 1 hora adelante (para HORIZON=1)
- `load+2`: valor 2 horas adelante (para HORIZON=2)
- `load+3`: valor 3 horas adelante (para HORIZON=3)

Así el loop puede comparar cada predicción con el valor real.

In [ ]:
%%time
training_window = 720  # 30 días (720 horas) de ventana de entrenamiento

train_ts = train['load']
test_ts = test_shifted

# Historial: últimos 720 valores del train
history = [x for x in train_ts]
history = history[(-training_window):]

predictions = list()

# Walk-forward loop
for t in range(test_ts.shape[0]):
    # 1. Entrenar modelo con historial actual
    model = SARIMAX(endog=history, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit()
    
    # 2. Predecir HORIZON pasos adelante
    yhat = model_fit.forecast(steps=HORIZON)
    predictions.append(yhat)
    
    # 3. Obtener valor real
    obs = list(test_ts.iloc[t])
    
    # 4. Mover ventana: agregar obs real, quitar el más viejo
    history.append(obs[0])
    history.pop(0)
    
    # 5. Log cada paso
    print(f'{test_ts.index[t]} | paso {t+1}: predicho={yhat[:2]}... real={obs[:2]}...')

## 7. Evaluar con MAPE

**MAPE** = Mean Absolute Percentage Error

```
MAPE = mean( |real - predicho| / real ) × 100%
```

- MAPE = 0.5% → predije con 0.5% de error (muy bueno)
- MAPE = 2% → predije con 2% de error (bueno)
- MAPE = 10% → predije con 10% de error (malo)

In [ ]:
# Crear DataFrame de evaluación
eval_df = pd.DataFrame(predictions, columns=['t+'+str(t) for t in range(1, HORIZON+1)])
eval_df['timestamp'] = test.index[0:len(test.index)-HORIZON+1]
eval_df = pd.melt(eval_df, id_vars='timestamp', value_name='prediction', var_name='h')
eval_df['actual'] = np.array(np.transpose(test_ts)).ravel()

# Invertir escala: de [0,1] a MW originales
eval_df[['prediction', 'actual']] = scaler.inverse_transform(eval_df[['prediction', 'actual']])

eval_df.head()

In [ ]:
# MAPE por horizonte (t+1, t+2, t+3)
if HORIZON > 1:
    eval_df['APE'] = (eval_df['prediction'] - eval_df['actual']).abs() / eval_df['actual']
    print('MAPE por horizonte:')
    print(eval_df.groupby('h')['APE'].mean())
    print()

In [ ]:
# One-step MAPE (solo t+1)
one_step_mape = mape(
    eval_df[eval_df['h'] == 't+1']['prediction'],
    eval_df[eval_df['h'] == 't+1']['actual']
) * 100

print(f'One-step forecast MAPE: {one_step_mape:.2f}%')

In [ ]:
# Multi-step MAPE (todos los horizontes)
multi_step_mape = mape(eval_df['prediction'], eval_df['actual']) * 100
print(f'Multi-step forecast MAPE: {multi_step_mape:.2f}%')
print()
if multi_step_mape < 2:
    print('Excelente: menos de 2% de error')
elif multi_step_mape < 5:
    print('Bueno: menos de 5% de error')
else:
    print('Regular: más de 5% de error — hay que mejorar')

## 8. Visualizar predicciones vs reales

In [ ]:
if HORIZON == 1:
    eval_df.plot(x='timestamp', y=['actual', 'prediction'],
                 style=['r', 'b'], figsize=(15, 8))
else:
    # Preparar datos para gráfico multi-step
    plot_df = eval_df[(eval_df.h=='t+1')][['timestamp', 'actual']]
    for t in range(1, HORIZON+1):
        plot_df['t+'+str(t)] = eval_df[(eval_df.h=='t+'+str(t))]['prediction'].values

    fig = plt.figure(figsize=(15, 8))
    ax = plt.plot(plot_df['timestamp'], plot_df['actual'],
                  color='red', linewidth=4.0, label='Actual')
    ax = fig.add_subplot(111)
    
    for t in range(1, HORIZON+1):
        x = plot_df['timestamp'][(t-1):]
        y = plot_df['t+'+str(t)][0:len(x)]
        ax.plot(x, y, color='blue',
                linewidth=4*math.pow(.9,t),
                alpha=math.pow(0.8,t),
                label=f'Predicción t+{t}')

    ax.legend(loc='best')

plt.xlabel('timestamp', fontsize=12)
plt.ylabel('load (MW)', fontsize=12)
plt.title(f'Predicción vs Real — MAPE: {multi_step_mape:.2f}%', fontsize=14)
plt.show()

## 9. Resumen

### Qué hicimos

| Paso | Qué hicimos | Por qué |
|------|-------------|--------|
| Split temporal | Train=Nov, Test=Dic | No mezclar futuro con pasado |
| Escalar | MinMaxScaler [0,1] | ARIMA funciona mejor con datos acotados |
| SARIMAX | order=(4,1,0), seasonal=(1,1,0,24) | Capturar tendencia + estacionalidad diaria |
| Walk-forward | Re-entrenar cada hora | Simular producción real |
| MAPE | Error porcentual | Medir calidad de predicción |

### Qué aprendimos

1. **ARIMA** = AutoRegresivo + Integrado + Moving Average
2. **SARIMAX** = ARIMA + componente estacional (S) + features exógenas (X)
3. **Walk-forward** = gold standard de validación en series temporales
4. **MAPE** = métrica principal para evaluar predicciones

---

**Siguiente**: [Lección 3: SVR](lesson-3-svr.md) — Modelo no-lineal alternativo